# Comparação de transformers para classificação de emails

Este notebook compara três checkpoints com fine-tuning supervisionado e a versão zero-shot do XNLI, sempre nas mesmas mensagens e nos mesmos cinco folds estratificados:

- fine-tuning supervisionado de `FacebookAI/xlm-roberta-base` com uma nova cabeça de três classes;
- fine-tuning supervisionado de `neuralmind/bert-base-portuguese-cased` com uma nova cabeça de três classes;
- fine-tuning supervisionado de `joeddav/xlm-roberta-large-xnli` com uma nova cabeça de três classes;
- classificação zero-shot de `joeddav/xlm-roberta-large-xnli` através de pares `texto + hipótese` e do logit de *entailment*.

Os modelos supervisionados recomeçam do checkpoint base em cada fold. A versão zero-shot não é treinada; os folds servem apenas para manter a mesma cobertura dos emails. Assim, a comparação mostra diretamente o efeito do fine-tuning no checkpoint XNLI. As únicas saídas visíveis são as duas tabelas finais.

In [7]:
import json
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.utils import logging as transformers_logging

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Executa o notebook na raiz do projeto ou dentro de research/.')
sys.path.insert(0, str(ROOT))

from src.email_data import LABELS
from src.finetuning import (
    DEFAULT_EMAILS_DIR,
    DEFAULT_EXCEL,
    EmailDataset,
    clear_device_cache,
    load_examples,
    make_model,
    scores,
    stratified_folds,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

if not torch.cuda.is_available():
    raise RuntimeError('Este notebook requer uma GPU NVIDIA com CUDA.')

DEVICE = torch.device('cuda')
USE_BF16 = torch.cuda.is_bf16_supported()
SEED = 42
N_FOLDS = 5
EPOCHS = 5
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
MAX_LENGTH = 512

SUPERVISED_MODELS = {
    'XLM-R base (fine-tuning)': 'FacebookAI/xlm-roberta-base',
    'BERTimbau base (fine-tuning)': 'neuralmind/bert-base-portuguese-cased',
    'XLM-R large XNLI (fine-tuning)': 'joeddav/xlm-roberta-large-xnli',
}
XNLI_DISPLAY_NAME = 'XLM-R large XNLI (zero-shot)'
XNLI_MODEL_NAME = 'joeddav/xlm-roberta-large-xnli'
LABEL_HYPOTHESES = {
    'Pedido de Informação': 'Este email contém um pedido de informação, preço, orçamento ou esclarecimento.',
    'Pedido de Encomenda': 'Este email contém uma encomenda ou uma confirmação explícita de compra.',
    'SPAM': 'Este email é spam, fraude, publicidade não solicitada ou conteúdo comercial irrelevante.',
}

OUTPUT_DIR = ROOT / 'data/model_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

In [8]:
examples = load_examples(DEFAULT_EXCEL, [DEFAULT_EMAILS_DIR])
folds = stratified_folds(examples, N_FOLDS, SEED)
fold_by_uid = {
    uid: fold_index
    for fold_index, fold in enumerate(folds, start=1)
    for uid, _text, _label in fold
}
assert len(fold_by_uid) == len(examples)
assert set(fold_by_uid) == {uid for uid, _text, _label in examples}

In [9]:
def training_arguments(output_dir, fold_seed):
    return TrainingArguments(
        output_dir=output_dir,
        use_cpu=False,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        eval_strategy='no',
        save_strategy='no',
        logging_strategy='no',
        report_to='none',
        disable_tqdm=True,
        fp16=not USE_BF16,
        bf16=USE_BF16,
        dataloader_pin_memory=True,
        seed=fold_seed,
        data_seed=fold_seed,
    )


def supervised_cross_validation(display_name, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collator = DataCollatorWithPadding(tokenizer)
    predictions = []

    for fold_index, test_examples in enumerate(folds, start=1):
        test_uids = {uid for uid, _text, _label in test_examples}
        train_examples = [example for example in examples if example[0] not in test_uids]
        fold_seed = SEED + fold_index - 1
        set_seed(fold_seed)

        train_dataset = EmailDataset(train_examples, tokenizer, MAX_LENGTH)
        test_dataset = EmailDataset(test_examples, tokenizer, MAX_LENGTH)
        with tempfile.TemporaryDirectory(prefix=f'comparison-fold-{fold_index}-') as temporary:
            # Em checkpoints base, make_model cria uma cabeça nova com três outputs.
            model = make_model(model_name)
            trainer = Trainer(
                model=model,
                args=training_arguments(temporary, fold_seed),
                train_dataset=train_dataset,
                data_collator=collator,
                processing_class=tokenizer,
            )
            trainer.train()
            output = trainer.predict(test_dataset)
            predicted_indices = np.argmax(output.predictions, axis=-1)

            for uid, real_label, predicted_index in zip(
                test_dataset.uids, test_dataset.labels, predicted_indices
            ):
                predictions.append({
                    'model': display_name,
                    'uid': uid,
                    'fold': fold_index,
                    'label_real': real_label,
                    'label_prevista': LABELS[int(predicted_index)],
                })

            del output, trainer, model, train_dataset, test_dataset
            clear_device_cache()

    assert len(predictions) == len(examples)
    assert len({row['uid'] for row in predictions}) == len(examples)
    return sorted(predictions, key=lambda row: row['uid'])


def entailment_label_id(model):
    for name, index in model.config.label2id.items():
        if str(name).casefold().startswith('entail'):
            return int(index)
    for index, name in model.config.id2label.items():
        if str(name).casefold().startswith('entail'):
            return int(index)
    raise ValueError('O checkpoint XNLI não identifica o logit de entailment.')


def xnli_zero_shot_predictions():
    tokenizer = AutoTokenizer.from_pretrained(XNLI_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        XNLI_MODEL_NAME,
        dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    ).to(DEVICE)
    model.eval()
    entailment_id = entailment_label_id(model)

    pair_texts = []
    pair_hypotheses = []
    for _uid, text, _label in examples:
        for label in LABELS:
            pair_texts.append(text)
            pair_hypotheses.append(LABEL_HYPOTHESES[label])

    entailment_scores = []
    with torch.inference_mode():
        for start in range(0, len(pair_texts), EVAL_BATCH_SIZE):
            batch = tokenizer(
                pair_texts[start:start + EVAL_BATCH_SIZE],
                pair_hypotheses[start:start + EVAL_BATCH_SIZE],
                padding=True,
                truncation='only_first',
                max_length=MAX_LENGTH,
                return_tensors='pt',
            ).to(DEVICE)
            logits = model(**batch).logits[:, entailment_id]
            entailment_scores.extend(logits.float().cpu().tolist())

    score_matrix = np.asarray(entailment_scores).reshape(len(examples), len(LABELS))
    predicted_indices = score_matrix.argmax(axis=1)
    predictions = [
        {
            'model': XNLI_DISPLAY_NAME,
            'uid': uid,
            'fold': fold_by_uid[uid],
            'label_real': real_label,
            'label_prevista': LABELS[int(predicted_index)],
        }
        for (uid, _text, real_label), predicted_index
        in zip(examples, predicted_indices)
    ]

    del model, tokenizer, score_matrix
    clear_device_cache()
    return sorted(predictions, key=lambda row: row['uid'])

In [10]:
results_by_model = {
    display_name: supervised_cross_validation(display_name, model_name)
    for display_name, model_name in SUPERVISED_MODELS.items()
}
results_by_model[XNLI_DISPLAY_NAME] = xnli_zero_shot_predictions()

summary_rows = []
recall_rows = []
all_predictions = []

for model_name, rows in results_by_model.items():
    truth = [row['label_real'] for row in rows]
    predicted = [row['label_prevista'] for row in rows]
    model_scores = scores(truth, predicted)
    summary_rows.append({
        'Modelo': model_name,
        'Accuracy': model_scores['accuracy'],
        'Recall macro': model_scores['recall_macro'],
    })
    recall_rows.append({
        'Modelo': model_name,
        **{
            f'Recall — {label}': model_scores['per_class'][label]['recall']
            for label in LABELS
        },
    })
    all_predictions.extend(rows)

summary_table = pd.DataFrame(summary_rows).set_index('Modelo')
recall_table = pd.DataFrame(recall_rows).set_index('Modelo')
predictions_table = pd.DataFrame(all_predictions)

summary_table.to_csv(OUTPUT_DIR / 'summary.csv', encoding='utf-8-sig')
recall_table.to_csv(OUTPUT_DIR / 'recall_by_class.csv', encoding='utf-8-sig')
predictions_table.to_csv(OUTPUT_DIR / 'predictions.csv', index=False, encoding='utf-8-sig')
_ = (OUTPUT_DIR / 'configuration.json').write_text(
    json.dumps({
        'seed': SEED,
        'folds': N_FOLDS,
        'epochs': EPOCHS,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'max_length': MAX_LENGTH,
        'supervised_models': SUPERVISED_MODELS,
        'xnli_model': XNLI_MODEL_NAME,
        'label_hypotheses': LABEL_HYPOTHESES,
        'folds_by_uid': fold_by_uid,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

{'train_runtime': '3.745', 'train_samples_per_second': '22.7', 'train_steps_per_second': '12.02', 'train_loss': '1.107', 'epoch': '5'}
{'train_runtime': '3.767', 'train_samples_per_second': '22.57', 'train_steps_per_second': '11.95', 'train_loss': '1.052', 'epoch': '5'}
{'train_runtime': '3.814', 'train_samples_per_second': '23.6', 'train_steps_per_second': '11.8', 'train_loss': '1.093', 'epoch': '5'}
{'train_runtime': '3.831', 'train_samples_per_second': '23.49', 'train_steps_per_second': '11.75', 'train_loss': '1.104', 'epoch': '5'}
{'train_runtime': '3.819', 'train_samples_per_second': '23.57', 'train_steps_per_second': '11.79', 'train_loss': '1.08', 'epoch': '5'}
{'train_runtime': '2.106', 'train_samples_per_second': '40.35', 'train_steps_per_second': '21.36', 'train_loss': '0.9547', 'epoch': '5'}
{'train_runtime': '2.162', 'train_samples_per_second': '39.32', 'train_steps_per_second': '20.82', 'train_loss': '0.889', 'epoch': '5'}
{'train_runtime': '2.201', 'train_samples_per_secon

## Resultados globais

In [11]:
display(summary_table.style.format('{:.3f}'))

,Accuracy,Recall macro
Modelo,,
XLM-R base (fine-tuning),0.318,0.296
BERTimbau base (fine-tuning),0.364,0.315
XLM-R large XNLI (fine-tuning),0.318,0.280
XLM-R large XNLI (zero-shot),0.409,0.333


## Recall por classe

In [12]:
display(recall_table.style.format('{:.3f}'))

,Recall — Pedido de Informação,Recall — Pedido de Encomenda,Recall — SPAM
Modelo,,,
XLM-R base (fine-tuning),0.556,0.333,0.000
BERTimbau base (fine-tuning),0.778,0.167,0.000
XLM-R large XNLI (fine-tuning),0.556,0.000,0.286
XLM-R large XNLI (zero-shot),1.000,0.000,0.000
